In [0]:

# Bronze -> Silver for the `fund` source (internal CSV, 5 rows).


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F

### 1. Read Bronze

In [0]:
bronze_df = read_bronze(spark, "fund")
display(bronze_df)
print(f"Bronze row count: {bronze_df.count()}")

fund_id,fund_name,vintage_year,fund_size_usd,_run_id,_ingested_at
FUND_001,Approach Capital Fund 1,2023,4.81205454E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_002,Seek Capital Fund 2,2022,3.97261097E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_003,Experience Capital Fund 3,2020,1.71139448E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_004,Material Capital Fund 4,2022,4.48840983E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_005,Nation Capital Fund 5,2021,3.23025902E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_001,Approach Capital Fund 1,2023,4.81205454E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_002,Seek Capital Fund 2,2022,3.97261097E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_003,Experience Capital Fund 3,2020,1.71139448E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_004,Material Capital Fund 4,2022,4.48840983E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z
FUND_005,Nation Capital Fund 5,2021,3.23025902E8,357da2da-cb85-426a-8c9d-374f6c4a58aa,2026-09-21T08:08:51.135Z


Bronze row count: 25


### 2. Type casting & text standardization
Rules : vintage_year/fund_size_usd -> numeric, trim/standardize text fields.

In [0]:
typed_df = (
    bronze_df
    .withColumn("fund_id", F.trim(F.col("fund_id")))
    .withColumn("fund_name", F.trim(F.col("fund_name")))
    .withColumn("vintage_year", F.col("vintage_year").cast("int"))
    .withColumn("fund_size_usd", F.col("fund_size_usd").cast("double"))
)

### 3. Null check
Required fields for `fund`: fund_id, fund_name, vintage_year, fund_size_usd
(all 4 columns are expected to always be populated - there is no
"optional" column on this dimension table).

In [0]:
REQUIRED_COLS = ["fund_id", "fund_name", "vintage_year", "fund_size_usd"]

clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)

null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, "fund")
print(f"Rows failing null check: {null_reject_count}")

Rows failing null check: 0


### 4. Duplicate check
Business key = fund_id (primary key of this table). Compare columns =
everything else - an exact repeat is a true duplicate; a repeated
fund_id with different attribute values is a break worth flagging
(shouldn't happen on this table, but the check is cheap and consistent
with every other source).

In [0]:
KEY_COLS = ["fund_id"]
COMPARE_COLS = ["fund_name", "vintage_year", "fund_size_usd"]

deduped_df, duplicates_df, breaks_df = split_duplicates(clean_df, KEY_COLS, COMPARE_COLS)

dup_count = duplicates_df.count()
break_count = breaks_df.count()

if dup_count > 0:
    write_quarantine(duplicates_df, "fund")
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("FUND_ATTRIBUTE_BREAK")), "fund")

print(f"Exact duplicates: {dup_count} | Conflicting-value breaks: {break_count}")

Exact duplicates: 20 | Conflicting-value breaks: 0


### 5. Write to Silver

In [0]:
write_silver(deduped_df, "fund")
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 5


### 6. Log to the shared DQ table

In [0]:
from datetime import date

business_date_str = date.today().isoformat()   # or pass in as a notebook parameter for a specific run date

log_dq(spark, "fund", business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, "fund", business_date_str, "duplicate_record", clean_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, "fund", business_date_str, "attribute_break", clean_df.count(), break_count, "FUND_ATTRIBUTE_BREAK")

print("DQ log written.")

/home/spark-5f42c30f-3aac-4aa1-b73b-33/.ipykernel/71/command-5696143635338715-3986853518:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


DQ log written.


### 7. Sanity check
Bronze count should equal Silver count + quarantined count (nothing
silently lost or double-counted).

In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + dup_count  # breaks are NOT removed from Silver, only flagged

assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, "
    f"silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=25 = silver=5 + quarantined=20
